In [1]:
!pip install underthesea tokenizers -q 
!pip install pandarallel -q 
!pip install pyarrow -q 
!pip install evaluate rouge_score -q
!pip install torchinfo -q 
!pip install pyngrok -q
!pip install bert_score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 105.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have nu

In [2]:
from underthesea import word_tokenize
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)
import pandas as pd
import os

# Standard library
import os
import random
import math
import copy
import multiprocessing
import time
import subprocess

# Data & utils
import numpy as np
import pandas as pd
from tqdm import tqdm
import gc

# Underthesea & tokenizers
from underthesea import word_tokenize
from pandarallel import pandarallel
from pandarallel.core import WorkerStatus
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace
from tokenizers.decoders import Metaspace as MetaspaceDecoder
import torch

# Ngrok & Kaggle
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os
import gc
import re
import time
import json
import random
import numpy as np
import pandas as pd
import requests
from tqdm import tqdm
from IPython.display import display
from kaggle_secrets import UserSecretsClient

def seed(seed_value=42):
  os.environ['PYTHONHASHSEED'] = str(seed_value)
  torch.manual_seed(seed_value)
  random.seed(seed_value)
  np.random.seed(seed_value)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
  print('Done')

def seed_worker(worker_id):
  worker_seed = torch.initial_seed() % 2**32
  np.random.seed(worker_seed)
  random.seed(worker_seed)

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [3]:
import pandas as pd
df_master = pd.read_csv('/kaggle/input/datasets/longnguyen2k5/test-set-prediction-results/master_evaluation_dataset.csv')
model_columns = [col for col in df_master.columns if col not in ['source', 'reference']]

In [4]:
# =====================================================================
# NOTEBOOK 1: TIẾN TRÌNH TÍNH TOÁN ĐỘ ĐO TỰ ĐỘNG CỤC BỘ (LOCAL METRICS)
# =====================================================================
from torchinfo import summary
import os
import re
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import evaluate
from IPython.display import display, HTML

# Thiết lập hạt giống ngẫu nhiên toàn cục để bảo toàn tính đồng bộ
def seed(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    print('Done')
seed(42)

# 1. ĐỌC FILE MASTER DATASET (ĐÃ CHẠY INFERENCE ĐẦY ĐỦ TẬP TEST)
# Giả sử file master đã được tạo từ pha generate summary trước đó
df_master = pd.read_csv('/kaggle/input/datasets/longnguyen2k5/test-set-prediction-results/master_evaluation_dataset.csv')
model_columns = [col for col in df_master.columns if col not in ['source', 'reference']]

# =====================================================================
# 2. HỆ THỐNG CÁC HÀM TOÁN HỌC ĐO ĐẠC NỘI TẠI (LOCAL ENGINES)
# =====================================================================
def calculate_intra_distinct(predictions, n_gram=2): 
    if len(predictions) == 0: 
        return 0.0
    distinct_score = 0.
    for text in predictions: 
        words = text.split()
        tup_list = [tuple(words[i:i+n_gram]) for i in range(len(words) - n_gram + 1)]
        distinct = set(tup_list)
        if len(tup_list) > 0: 
            distinct_score += len(distinct) / len(tup_list)
    return distinct_score / len(predictions)
    
def calculate_inter_distinct(predictions, n_gram=2): 
    distinct = set()
    total_nums = 0
    for text in predictions: 
        words = text.split()
        tup_list = [tuple(words[i:i+n_gram]) for i in range(len(words) - n_gram + 1)]
        total_nums += len(tup_list)
        distinct.update(tup_list)
    return len(distinct) / total_nums if total_nums > 0 else 0.0

def calculate_length_ratio(predictions, references):
    total_ratio = 0.0
    for pred, ref in zip(predictions, references):
        len_pred = len(pred.split())
        len_ref = len(ref.split())
        if len_ref > 0:
            total_ratio += len_pred / len_ref
    return total_ratio / len(predictions) if len(predictions) > 0 else 0.0

def calculate_novel_ngrams_score(predictions, sources, n_gram=1):
    if len(predictions) == 0: return 0.0
    total_novel_ratio = 0.0
    for pred, src in zip(predictions, sources):
        pred_words = pred.split()
        src_words = src.split()
        
        src_ngrams = set([
            tuple(src_words[i:i+n_gram]) 
            for i in range(len(src_words) - n_gram + 1)
        ])
        pred_ngrams = [
            tuple(pred_words[i:i+n_gram]) 
            for i in range(len(pred_words) - n_gram + 1)
        ]
        
        if len(pred_ngrams) == 0: continue
        novel_count = sum(1 for tg in pred_ngrams if tg not in src_ngrams)
        total_novel_ratio += novel_count / len(pred_ngrams)
        
    return (total_novel_ratio / len(predictions)) * 100
    
def calculate_entity_precision_local(predictions, sources):
    """Tính Named Entity Precision bằng Regex làm độ đo đối chứng cục bộ"""
    if len(predictions) == 0: return 0.0
    total_precision = 0.0
    proper_noun_regex = r'\b[A-Z-ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĐ][a-z-àáâãèéêìíòóôõùúýđ]*(?:\s+[A-Z-ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĐ][a-z-àáâãèéêìíòóôõùúýđ]*)*\b'
    for pred, src in zip(predictions, sources):
        pred_entities = set(re.findall(proper_noun_regex, pred))
        pred_entities = {e for e in pred_entities if len(e.split()) > 1 or e.lower() not in ['khu', 'nhà', 'đây', 'bài', 'với', 'trong']}
        if len(pred_entities) == 0:
            total_precision += 1.0
            continue
        correct_entities = sum(1 for entity in pred_entities if entity in src)
        total_precision += correct_entities / len(pred_entities)
    return (total_precision / len(predictions)) * 100
    
def calculate_evalute_metrics(predictions, references, sources=None): 
    final_results = {}
    final_results['intra_distinct'] = calculate_intra_distinct(predictions)
    final_results['inter_distinct'] = calculate_inter_distinct(predictions)
    final_results['len_ratio'] = calculate_length_ratio(predictions, references)
    final_results['novel_1gram'] = calculate_novel_ngrams_score(predictions, sources, n_gram=1)
    final_results['novel_2gram'] = calculate_novel_ngrams_score(predictions, sources, n_gram=2)
    final_results['entity_precision_local'] = calculate_entity_precision_local(predictions, sources)
    
    for metric in metrics_list:
        if metric.name in ["bertscore", "bert_score"]:
            # Ép xử lý phân đoạn batch_size=64 trên cấu hình phần cứng hiện tại
            raw_bert = metric.compute(predictions=predictions, references=references, lang="vi", device="cuda", batch_size=64)
            final_results["bertscore"] = sum(raw_bert["f1"]) / len(raw_bert["f1"])
        elif metric.name == 'rouge':
            raw_rouge = metric.compute(predictions=predictions, references=references)
            final_results['rouge1'] = raw_rouge['rouge1']
            final_results['rouge2'] = raw_rouge['rouge2']
            final_results['rougeL'] = raw_rouge['rougeL']
        elif metric.name == 'bleu':
            raw_bleu = metric.compute(predictions=predictions, references=references)
            final_results['bleu'] = raw_bleu['bleu']
    return final_results

# =====================================================================
# 3. KHỞI CHẠY TIẾN TRÌNH QUÉT VÀ TRÍCH XUẤT ĐIỂM SỐ CHÍNH THỨC
# =====================================================================
metrics_list = [
    evaluate.load('rouge'),
    evaluate.load('bleu'),
    evaluate.load('bertscore')
]

print(f"📊 Đang kích hoạt các độ đo tự động hàng loạt trên toàn bộ {len(df_master)} dòng...")
auto_dict_results = []

for col_name in tqdm(model_columns, desc="[AUTO METRICS ENGINES]"):
    preds = df_master[col_name].dropna().tolist()
    refs = df_master['reference'].dropna().tolist()
    srcs = df_master['source'].dropna().tolist()
    
    # Tính toán tập hợp chỉ số
    auto = calculate_evalute_metrics(preds, refs, srcs)
    
    # Gom dữ liệu phẳng theo hàng ngang để chuẩn bị đảo trục chuẩn IEEE
    row = {
        "Độ đo tự động (Local Metrics)": col_name,
        "ROUGE-1 (%)": auto['rouge1'] * 100,
        "ROUGE-2 (%)": auto['rouge2'] * 100,
        "ROUGE-L (%)": auto['rougeL'] * 100,
        "BLEU (%)": auto['bleu'] * 100,
        "BERTScore F1 (%)": auto['bertscore'] * 100,
        "Intra-Distinct (2-gram)": auto['intra_distinct'],
        "Inter-Distinct (2-gram)": auto['inter_distinct'],
        "Novel 1-gram (%)": auto['novel_1gram'],
        "Novel 2-gram (%)": auto['novel_2gram'],
        "Entity Precision (Regex) (%)": auto['entity_precision_local']
    }
    auto_dict_results.append(row)

# Đóng gói và quay trục Transpose: Đưa Metric thành hàng dọc, Tên 16 mô hình thành cột ngang
df_auto_raw = pd.DataFrame(auto_dict_results)
df_auto_ieee_style = df_auto_raw.set_index("Độ đo tự động (Local Metrics)").T

# =====================================================================
# 4. LƯU FILE CỨNG VÀ HIỂN THỊ UI BÁO CÁO LUNG LINH
# =====================================================================
# Lưu file CSV cố định ra làm việc để nộp thầy hoặc nhét sang Notebook vẽ hình
df_auto_ieee_style.to_csv('/kaggle/working/automatic_metrics_summary_ieee.csv', index=True)
print("\n💾 THÀNH CÔNG! Đã xuất và lưu bảng số liệu chính thức tại: /kaggle/working/automatic_metrics_summary_ieee.csv")

def display_ultimate_auto_report(df_report):
    format_config = {col: "{:.2f}" for col in df_report.columns}
    
    # Thuật toán tìm giá trị MAX theo hàng ngang (axis=1) để highlight ô đỉnh nhất của từng metric
    def highlight_max_row(s):
        is_max = s == s.max()
        return ['color: #27ae60; font-weight: bold; background-color: #f2f9f5;' if v else '' for v in is_max]
    
    styled_html = (df_report.style
        .format(format_config)
        .apply(highlight_max_row, axis=1) 
        .set_caption("🏆 BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ ĐO TỰ ĐỘNG TOÀN DIỆN (AUTOMATIC LOCAL METRICS)")
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('margin-bottom', '15px'), ('color', '#2c3e50'), ('text-align', 'center')]},
            {'selector': 'th', 'props': [('background-color', '#8e44ad'), ('color', 'white'), ('font-size', '11px'), ('padding', '10px'), ('text-align', 'center'), ('border', '1px solid #ddd')]},
            {'selector': 'td', 'props': [('font-size', '11px'), ('padding', '10px'), ('text-align', 'center'), ('border', '1px solid #ddd')]}
        ])
    )
    display(styled_html)

# Kích hoạt hiển thị ma trận kết quả bôi màu trực quan
display_ultimate_auto_report(df_auto_ieee_style)

Done


📊 Đang kích hoạt các độ đo tự động hàng loạt trên toàn bộ 1344 dòng...


[AUTO METRICS ENGINES]:   0%|          | 0/16 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[AUTO METRICS ENGINES]: 100%|██████████| 16/16 [10:24<00:00, 39.06s/it]


💾 THÀNH CÔNG! Đã xuất và lưu bảng số liệu chính thức tại: /kaggle/working/automatic_metrics_summary_ieee.csv


Độ đo tự động (Local Metrics),Baseline Transformer (Vanilla),Baseline Transformer (Penalty),Improved Baseline (Vanilla),Improved Baseline (Penalty),Soft Prompt Mamba (Vanilla),Soft Prompt Mamba (Penalty),Entity Gated Mamba (Vanilla),Entity Gated Mamba (Penalty),Transformer Hypersphere (Vanilla),Transformer Hypersphere (Penalty),Entity Gated Mamba With Label Smoothing (Vanilla),Entity Gated Mamba With Label Smoothing (Penalty),Entity Guided Hybrid Mamba (Vanilla),Entity Guided Hybrid Mamba (Penalty),Entity Guided Pure Transformer (Vanilla),Entity Guided Pure Transformer (Penalty)
ROUGE-1 (%),39.10,60.25,53.52,62.27,50.44,61.61,57.50,61.50,66.11,67.04,57.36,60.24,66.14,67.89,66.74,68.15
ROUGE-2 (%),11.77,22.12,17.34,22.77,15.42,21.99,19.21,21.95,34.01,34.83,19.12,21.15,33.04,34.61,34.57,35.96
ROUGE-L (%),22.65,30.48,28.88,30.97,27.54,30.44,30.22,30.85,39.24,39.67,30.10,30.40,38.14,38.56,39.59,40.07
BLEU (%),1.31,3.57,3.43,4.26,2.62,3.89,3.78,3.97,18.74,18.95,3.62,3.69,16.92,17.04,19.81,19.94
BERTScore F1 (%),62.10,70.12,67.56,70.76,66.59,70.58,69.02,70.34,74.55,74.94,69.02,69.94,74.20,74.88,74.78,75.41
Intra-Distinct (2-gram),0.34,0.93,0.60,0.93,0.56,0.96,0.74,0.93,0.81,0.84,0.76,0.91,0.79,0.88,0.80,0.87
Inter-Distinct (2-gram),0.05,0.19,0.11,0.18,0.08,0.17,0.10,0.13,0.34,0.36,0.11,0.13,0.34,0.38,0.35,0.38
Novel 1-gram (%),24.96,32.04,28.65,32.89,32.94,37.22,32.20,34.41,6.32,6.58,32.42,34.20,7.04,7.62,6.08,6.42
Novel 2-gram (%),75.77,79.66,77.12,79.67,80.61,83.02,80.55,81.99,30.55,30.90,80.83,81.90,35.07,35.46,29.56,29.44
Entity Precision (Regex) (%),46.55,40.97,52.94,48.54,42.75,39.39,47.19,45.26,86.48,85.98,47.66,44.50,85.47,83.93,86.44,85.65
